# Restarting nodes

The Rust runtime can create a new attempt without repeating unaffected work. With no locations, `restart_task` restarts all errored nodes. With explicit locations, it invalidates those nodes and their transitive dependants. The source attempt remains unchanged.


In [ ]:
from data.typed_eval import typed_eval
from tierkreis.runtime import new_default

runtime = await new_default()
with runtime:
    workflow_id = await runtime.save_workflow("restart_example", typed_eval())
    run_id = await runtime.start_new_run(workflow_id, {})
    await runtime.wait_for(run_id)
    original_outputs = await runtime.get_outputs(run_id)

If we open the workflow in the visualiser then we should see something like the following

<img src="../_static/restart_example_complete.png" alt="Completed workflow" />

which shows that all the nodes have finished.

Next we restart the task labelled "builtins.itimes" with the following code.
(Alternatively we can click on the node in the visualiser and press the 'Restart' button.)

In [ ]:
with runtime:
    new_attempt, invalidated = await runtime.restart_task(run_id, ["N3.N3"])

print(new_attempt, invalidated)

If we open the workflow in the visualiser again then we should see something like the following

<img src="../_static//restart_example_midway.png" alt="Completed workflow"  />

which shows that the chosen node has been restarted as well as all the nodes dependent on it.

Next we resume graph execution to complete all the invalidated nodes:

In [ ]:
with runtime:
    await runtime.wait_for(run_id, new_attempt)
    restarted_outputs = await runtime.get_outputs(run_id, new_attempt)
    assert await runtime.get_outputs(run_id) == original_outputs

assert restarted_outputs == original_outputs

The graph now shows as completed again, as in the first screenshot.